# E2B + Agents SDK: basic and updated Vite app

This notebook uses two fresh E2B sandboxes built from the same tiny Vite-style starter app.

- Sandbox 1 creates a quick basic version.
- Sandbox 2 creates a more polished updated version.

There is no preview server in this version. The notebook prints the summaries and the final `src/App.tsx` and `src/styles.css` from each sandbox.

In [ ]:
# Install the editable package and dotenv support. Run this notebook from the repo root.
%pip install -U -e ".[e2b]" python-dotenv

In [ ]:
# Import the SDK pieces, define the shared brief, and build a tiny Vite-style starter app.
import json
import os
from pathlib import Path

from dotenv import load_dotenv
from openai.types.responses import ResponseFunctionCallArgumentsDeltaEvent, ResponseTextDeltaEvent

from agents import ModelSettings, RawResponsesStreamEvent, RunItemStreamEvent, Runner
from agents.extensions.sandbox import E2BSandboxClient, E2BSandboxClientOptions, E2BSandboxType
from agents.run import RunConfig
from agents.sandbox import Manifest, SandboxAgent, SandboxRunConfig
from agents.sandbox.capabilities import ApplyPatch, Shell
from agents.sandbox.entries import File
from agents.sandbox.errors import WorkspaceReadNotFoundError

load_dotenv()

MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4-mini")
TIMEOUT_SECONDS = 1800
PREVIEW_PORT = 4173
ACTIVE_SESSIONS = []
BRIEF = (
    "E2B gives AI agents secure cloud sandboxes to run code, browse, and build things safely. "
    "The landing page should feel premium and credible instead of generic SaaS. "
    "Primary CTA: Start building."
)

missing = [name for name in ("OPENAI_API_KEY", "E2B_API_KEY") if not os.getenv(name)]
if missing:
    raise RuntimeError(f"Missing required environment variables: {', '.join(missing)}")


# package.json is easier to maintain as a normal Python dict.
PACKAGE_JSON = (
    json.dumps(
        {
            "name": "e2b-example",
            "private": True,
            "version": "0.0.0",
            "type": "module",
            "scripts": {
                "dev": "vite",
                "build": "vite build",
            },
            "dependencies": {
                "react": "^18.3.1",
                "react-dom": "^18.3.1",
            },
            "devDependencies": {
                "vite": "^5.4.0",
            },
        },
        indent=2,
    )
    + "\n"
)

# The other starter files are kept as short source strings so the example stays readable.
INDEX_HTML = """<!doctype html>
<html lang=\"en\">
  <head>
    <meta charset=\"UTF-8\" />
    <meta name=\"viewport\" content=\"width=device-width, initial-scale=1.0\" />
    <title>E2B</title>
  </head>
  <body>
    <div id=\"root\"></div>
    <script type=\"module\" src=\"/src/main.tsx\"></script>
  </body>
</html>
"""

MAIN_TSX = """import React from \"react\";
import ReactDOM from \"react-dom/client\";
import App from \"./App\";

ReactDOM.createRoot(document.getElementById(\"root\")!).render(
  <React.StrictMode>
    <App />
  </React.StrictMode>
);
"""

APP_TSX = """import React from \"react\";

export default function App() {
  return <main>E2B starter app.</main>;
}
"""

VITE_CONFIG_JS = """import { defineConfig } from \"vite\";

export default defineConfig({
  server: {
    host: \"0.0.0.0\",
    allowedHosts: [\".e2b.app\"],
  },
});
"""


# Turn the starter files into a manifest that each fresh sandbox will receive.
def vite_starter_manifest() -> Manifest:
    return Manifest(
        entries={
            "package.json": File(content=PACKAGE_JSON.encode()),
            "index.html": File(content=INDEX_HTML.encode()),
            "vite.config.js": File(content=VITE_CONFIG_JS.encode()),
            "src/main.tsx": File(content=MAIN_TSX.encode()),
            "src/App.tsx": File(content=APP_TSX.encode()),
        }
    )


# Build one sandbox agent for a single version prompt.
def build_agent(*, version_name: str, version_prompt: str, manifest: Manifest) -> SandboxAgent:
    return SandboxAgent(
        name=f"E2B Vite {version_name.title()} Builder",
        model=MODEL,
        instructions=(
            "Update the Vite app in the sandbox workspace. Edit src/App.tsx and any supporting "
            "directly and return a concise summary of what changed."
        ),
        developer_instructions=(
            f"Homepage brief: {BRIEF}\n"
            f"Version goal: {version_prompt}\n"
            "Start from the tiny Vite starter. You may create src/styles.css if you want, "
            "or keep styles inline in src/App.tsx. "
            "Do not add dependencies. Do not run npm install. "
            "Keep the result compatible with the starter Vite structure. "
            "Use apply_patch for edits and shell for quick inspection if needed."
        ),
        default_manifest=manifest,
        capabilities=[ApplyPatch(), Shell()],
        model_settings=ModelSettings(tool_choice="required"),
    )


# Read files back out of the sandbox so we can compare what each version produced.
async def _read_text(session, path: str) -> str:
    payload = await session.read(Path(path))
    data = payload.read()
    if isinstance(data, bytes):
        return data.decode("utf-8")
    return str(data)


async def _maybe_read_text(session, path: str) -> str | None:
    try:
        return await _read_text(session, path)
    except WorkspaceReadNotFoundError:
        return None


# Stream tool calls and model text so the notebook does not look frozen while it runs.
async def print_stream(result) -> None:
    text_open = False
    tool_args_open = False
    print("[run] streaming agent activity")
    async for event in result.stream_events():
        if isinstance(event, RawResponsesStreamEvent):
            data = event.data
            if isinstance(data, ResponseTextDeltaEvent):
                if tool_args_open:
                    print()
                    tool_args_open = False
                if not text_open:
                    print("[model] ", end="", flush=True)
                    text_open = True
                print(data.delta, end="", flush=True)
            elif isinstance(data, ResponseFunctionCallArgumentsDeltaEvent):
                if text_open:
                    print()
                    text_open = False
                if not tool_args_open:
                    print("[tool-args] ", end="", flush=True)
                    tool_args_open = True
                print(data.delta, end="", flush=True)
            continue

        if text_open:
            print()
            text_open = False
        if tool_args_open:
            print()
            tool_args_open = False

        if isinstance(event, RunItemStreamEvent):
            item_type = getattr(event.item, "type", None)
            if item_type == "tool_call_item":
                tool_name = getattr(event.item.raw_item, "name", "tool")
                print(f"[tool] calling {tool_name}")
            elif item_type == "tool_call_output_item":
                print("[tool] completed")


# Start a Vite dev server and print the public preview URL for this sandbox.
async def start_preview(session, version_name: str) -> str:
    print(f"[preview:{version_name}] starting Vite dev server")
    result = await session.exec(
        "sh",
        "-lc",
        (
            "npm install >/tmp/e2b-example-npm-install.log 2>&1 && "
            "nohup npm run dev -- --host 0.0.0.0 --port 4173 "
            ">/tmp/e2b-example-vite.log 2>&1 &"
        ),
        shell=False,
        timeout=120,
    )
    if not result.ok():
        raise RuntimeError(f"Failed to start preview server: {result.stderr}")
    endpoint = await session.resolve_exposed_port(PREVIEW_PORT)
    preview_url = endpoint.url_for("http")
    for _ in range(30):
        probe = await session.exec(
            "sh",
            "-lc",
            f"curl -fsS http://127.0.0.1:{PREVIEW_PORT}/ >/dev/null",
            shell=False,
            timeout=5,
        )
        if probe.ok():
            break
        await __import__("asyncio").sleep(1)
    else:
        raise RuntimeError("Preview server did not become ready in time")
    print(f"[preview:{version_name}] {preview_url}")
    return preview_url


# Create a brand-new sandbox for one version, run the agent, and capture the final files.
async def run_version(version_name: str, version_prompt: str) -> dict[str, str]:
    print(f"[setup:{version_name}] creating sandbox")
    manifest = vite_starter_manifest()
    session = await E2BSandboxClient().create(
        manifest=manifest,
        options=E2BSandboxClientOptions(
            sandbox_type=E2BSandboxType.E2B,
            timeout=TIMEOUT_SECONDS,
            exposed_ports=(PREVIEW_PORT,),
            allow_internet_access=True,
            pause_on_exit=True,
        ),
    )
    try:
        print(f"[setup:{version_name}] sandbox created: {session.state.sandbox_id}")
        await session.start()
        print(f"[setup:{version_name}] sandbox session started")

        agent = build_agent(
            version_name=version_name,
            version_prompt=version_prompt,
            manifest=manifest,
        )
        run_config = RunConfig(
            sandbox=SandboxRunConfig(
                client=None,
                session=session,
                options=None,
            )
        )

        result = Runner.run_streamed(
            agent,
            f"Make the {version_name} version now.",
            run_config=run_config,
        )
        await print_stream(result)

        app_tsx = await _read_text(session, "src/App.tsx")
        styles_css = await _maybe_read_text(session, "src/styles.css")
        preview_url = await start_preview(session, version_name)
        ACTIVE_SESSIONS.append(session)
        print(f"[done:{version_name}] captured final files")
        return {
            "sandbox_id": session.state.sandbox_id,
            "preview_url": preview_url,
            "summary": str(result.final_output),
            "app_tsx": app_tsx,
            "styles_css": styles_css or "<no src/styles.css created>",
        }
    except Exception:
        try:
            await session.shutdown()
        except Exception:
            pass
        raise

In [ ]:
# Run the basic and updated versions in separate sandboxes, then print their outputs.
basic = await run_version(  # type: ignore[top-level-await]  # noqa: F704
    "basic",
    ("Create a quick, clean first-pass landing page with tighter copy and minimal visual polish."),
)

updated = await run_version(  # type: ignore[top-level-await]  # noqa: F704
    "updated",
    (
        "Create a more distinctive, premium version with stronger typography, "
        "a clearer visual idea, and more confident product positioning."
    ),
)

print("\n=== Basic Summary ===\n")
print(basic["summary"])
print("\n=== Basic Preview URL ===\n")
print(basic["preview_url"])
print("\n=== Basic src/App.tsx ===\n")
print(basic["app_tsx"])
print("\n=== Basic src/styles.css ===\n")
print(basic["styles_css"])

print("\n=== Updated Summary ===\n")
print(updated["summary"])
print("\n=== Updated Preview URL ===\n")
print(updated["preview_url"])
print("\n=== Updated src/App.tsx ===\n")
print(updated["app_tsx"])
print("\n=== Updated src/styles.css ===\n")
print(updated["styles_css"])